# Import PyTerrier

In [1]:
import os
os.environ["JAVA_HOME"] = "C:\Program Files\Java\jdk-25.0.2"
#os.environ["JVM_PATH"] = "C:\Program Files\Java\jdk-25.0.2\lib\jvm.lib"

import pyterrier as pt
#pt.java.init()


import os
import json
import pandas as pd
#import pyterrier as pt

<>:2: SyntaxWarning: invalid escape sequence '\P'
<>:2: SyntaxWarning: invalid escape sequence '\P'
C:\Users\krist\AppData\Local\Temp\ipykernel_43336\4163070893.py:2: SyntaxWarning: invalid escape sequence '\P'
  os.environ["JAVA_HOME"] = "C:\Program Files\Java\jdk-25.0.2"


# Load Data

In [ ]:
import os
import json
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer  # ADD THIS

nltk.download('stopwords')
nltk.download('punkt')
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()  # ADD THIS

def remove_stop_words(text):
    stop_words = set(stopwords.words('english'))
    tokens = word_tokenize(text.lower())
    filtered_tokens = [stemmer.stem(word) for word in tokens if word not in stop_words]  # CHANGE THIS
    return (' '.join(filtered_tokens))

def load_mood_words_list(path):
    mood_list = []

    for filename in os.listdir(path):
        with open(os.path.join(path, filename), "r", encoding="utf-8") as file:
            data = json.load(file)

            # If the JSON file contains a list
            if isinstance(data, list):
                moods = data
            else:
                moods = [data]

            for mood in moods:
                for second in mood["secondary"]:
                    feeling = second.get("feeling", "")
                    description = second.get("description", "")
                    scenario = second.get("scenario", "")
                    category = second.get("category", "")
                    physical_sensations = ""
                    for sensation in second.get("physical_sensations", ""):
                        physical_sensations = f"{physical_sensations} {sensation}"
                    full_text = f"{feeling} {description} {scenario} {category} {physical_sensations}"

                    mood_list.append(remove_stop_words(full_text.strip("\"")))
                    for third in second.get("tertiary", []):                        
                        feeling2 = third.get("feeling", "")
                        description2 = third.get("description", "")
                        scenario2 = third.get("scenario", "")
                        category2 = third.get("category", "")
                        physical_sensations2 = ""
                        for sensation in second.get("physical_sensations", ""):
                            physical_sensations2 = f"{physical_sensations2} {sensation}"
                        full_text2 = f"{feeling2} {description2} {scenario2} {category2} {physical_sensations2}"
                        print(remove_stop_words(full_text2.strip("\"")))

                        mood_list.append(full_text2)

    return mood_list

mood_path = "./mood_data"
list_mood = load_mood_words_list(mood_path)
df_print = pd.DataFrame(list_mood, columns=["mood"])
#df_print.to_csv("moods2_stem.csv", index=False)

MPD_PATH = "./mpd_data"

def load_songs(path):
    df = []

    for filename in os.listdir(path):
        with open(os.path.join(path, filename), "r", encoding="utf-8") as file:
            data = json.load(file)

            for playlist in data["playlists"]:
                title = playlist.get("name", "")
                title_arr = title.split()
                for txt in list_mood:
                    for tok in title_arr:
                        if stemmer.stem(tok.lower()) in txt.lower():
                            title = f"{title} {txt}"

                tracks = playlist.get("tracks", "") # "" as a default value
                track_text = ""
                for track in tracks: 
                    docno = f"{str(playlist["pid"])}_{str(track.get("pos", ""))}"
                    track_name = track.get("track_name", "")
                    artist_name = track.get("artist_name", "")
                    album_name = track.get("album_name", "")
                    track_text = (f"{track_name} {artist_name} {album_name}")

                    full_text = title + " " + " ".join(track_text)

                    df.append({
                        "docno": docno,
                        "text": full_text,
                        "song": f"{track_name} by {artist_name}",
                        "followers" : playlist["num_followers"],
                        "object" : track
                    })

    return pd.DataFrame(df)

df = load_songs(MPD_PATH)
df.to_csv("songs_expanded_final.csv", index=False)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\krist\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\krist\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


arous experienc excit eager , often sens heighten energi . posit giggl bounc energi relax postur
cheeki play bold impud , often charm teas way . posit giggl bounc energi relax postur
free feel unburden liber constraint worri . posit relax muscl soft breath warmth
joy experienc intens happi delight , often sens elat . posit relax muscl soft breath warmth
curiou strong desir know learn someth new . posit lean forward focus gaze slight smile
inquisit eager investig ask question , often show probe natur . posit lean forward focus gaze slight smile
success feel accomplish achiev goal receiv recognit . posit chest puf warm glow big smile
confid strong belief one 's abil worth , often sens self-assur . posit chest puf warm glow big smile
respect admir held high regard other . posit relax smile warmth chest calm breath
valu feel appreci import other , often person profession context . posit relax smile warmth chest calm breath
courag show braveri willing face challeng fear . posit strong postu

# Indexing

In [ ]:
base = os.path.abspath("var_song_five_expanded_final/index") # Append new folders onto absolute path
os.makedirs(base, exist_ok=True) 

indexer = pt.index.IterDictIndexer(base, overwrite=True)
index_ref = indexer.index(df.to_dict(orient="records"))
index = pt.IndexFactory.of(index_ref)


Java started (triggered by TerrierIndexer.__init__) and loaded: pyterrier.java.colab, pyterrier.java, pyterrier.java.24, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]


02:44:48.112 [main] WARN org.terrier.structures.indexing.Indexer -- Adding an empty document to the index (23_0) - further warnings are suppressed
02:48:01.169 [main] WARN org.terrier.structures.indexing.Indexer -- Indexed 19687 empty documents


Make a search

In [4]:
query = "energy"
bm25 = pt.terrier.Retriever(index, wmodel="BM25")
results = bm25.search(query)
results.sort_values

#sorted_df = df.sort_values(by='followers', ascending=True)
#print(sorted_df.iloc[results.head(30).docid].song)
top_30 = df.iloc[results.head(30).docid].song
